In [ ]:
import os
from dotenv import load_dotenv
import redcap
import pandas as pd
import pandasql as psql
from datetime import datetime, timedelta
import re

# Load environment variables
load_dotenv()

# Initialize REDCap projects
df = redcap.Project(
    os.getenv('REDCAP_MAIN_URL'),
    os.getenv('REDCAP_MAIN_TOKEN')
)

ck_wk = redcap.Project(
    os.getenv('REDCAP_CK_WK_URL'),
    os.getenv('REDCAP_CK_WK_TOKEN')
)

indigo = redcap.Project(
    os.getenv('REDCAP_INDIGO_URL'),
    os.getenv('REDCAP_INDIGO_TOKEN')
)

In [4]:
sen_data=ck_wk.export_records(forms=['enumeration_and_sensitisation'])
sen_log_data=pd.DataFrame(sen_data)
sen_log_data=sen_log_data[['wk_ckno','enu_name','enu_village', 'enu_mname', 'enu_fname','enu_comp','sen_contact1','sen_contact2','sen_contact3']]
sen_log_data=sen_log_data.rename(columns={'wk_ckno': 'ck_wkno'})
village_mapping = {
'001':	'DUMBUTO',
'002':	'SANKANDI',
'003':	'NIORO JATTABA',
'004':	'JATTABA',
'005':	'JIFFARONG',
'006':	'BAJANA',
'007':	'KULI KUNDA',
'008':	'JAMARU',
'009':	'BRIKAMANDING',
'010':	'KANTONG KUNDA',
'011':	'JALI',
'013':	'MANDUAR',
'014':	'BANG KULING',
'015':	'GISSAY',
'016':	'TANKULAR',
'017':	'JOLI',
'018':	'KUYANG',
'019':	'BANTASU',
'020':	'SANTAMBA',
'021':	'MISSIRA',
'022':	'TABORANGKOTO',
'023':	'BURONG',
'024':	'JULA KUNDA',
'025':	'KARANTABA',
'026':	'MANDINA',
'027':	'JANNEH KUNDA',
'028':	'KEMOTO',
'029':	'KENEBA',
'030':	'BATELLING',
'031':	'SANDENG',
'032':	'WUDEBA',
'034':	'KENOKOTO',
'035':	'MANARI',
'036':	'NINETEEN',
'040':	'WUROKANG',
'041':	'KWINELLA SANSANKONO',
'042':	'KWINELLA NIA KUNDA',
'043':	'TENDABA',
'044':	'BUMARR',
'045':	'BAMBAKO',
'046':	'KUNDONG MARIAYA',
'047':	'NEMA',
'048':	'KUNDANG NUMU KUNDA',
'049':	'KUNDANG FULA KUNDA',
'050':	'NEMA KUTA',
'051':	'JIRROFF',
'052':	'MADINA ANGALLEH',
'053':	'JATTA KUNDA',
'054':	'MANDINA CENTRAL',
'055':	'SARE SARJO',
'056':	'SIBETO',
'057':	'SARE NDALLA',
'058':	'TABANANI',
'060':	'WILLINGARA',
'061':	'SARE MAMUDU',
'070':	'KOLIOR',
'071':	'JOMARR',
'072':	'JASOBO',
'073':	'SARE MUSA',
'074':	'YORRO JULA',
'075':	'BABOU YAA',
'076':	'MASEMBEH',
'077':	'GENIERE',
'078':	'KAIAF',
'079':	'MADINA KAIAF',
'080':	'SARE SAMBA',
'081':	'MADINA CEESAY KUNDA',
'082':	'NJOLFEN',
'083':	'TORANKA BANTANG',
'084':	'SAREH PATEH',
'085':	'MUNKUTALA'
}
# DataFrame
sen_log_data['village_name'] = sen_log_data['enu_village'].map(village_mapping)
sen_log_data=sen_log_data[
    (sen_log_data['ck_wkno']!='')
]

In [5]:
clustermap={
   ('051', '050', '070', '054','052', '048', '053', '061', '055','049' ) : 'Nema Kuta',
   ('047', '045'):'Nema',
   ('041', '043', '030', '044', '060' ):'Kwinella Sansankono',
   ('042', '040',):'Kwinella Nia-Kunda',
   ('001', '032', '002', '004'):'Dumbuto',
   ('003','031',):'Niorro Jattaba',
   ('005', '008', '006', '009'):'Jiffarong',
   ('007', '010', '011' ):'Kuli Kunda',
   ('029',):'keneba',
   ('013', '016', '015', '019','014'):'Manduar',
   ('025', '018', '026', '024', '021', '022', '023'):'Karantaba',
   ('028', '027','017',):'Kemoto',
   ('076', '081', '077', '074', '075', '073' ):'MASSEMBEH',
   ('078', '079', '080' ):'KAIAF',
   ('071', '072', '083', '084', '082'):'Kolior'

}


def map_village_to_cluster(village_code):
    for key, cluster in clustermap.items():
        if village_code in key:
            return cluster
    return 'Unknown'  # Default value if the village code is not found

# Apply this function to create the 'cluster_mapping' column
sen_log_data['cluster_mapping'] = sen_log_data['enu_village'].apply(map_village_to_cluster)
sen_log_data=sen_log_data[['ck_wkno', 'enu_name', 'enu_comp', 'enu_mname', 'enu_fname','sen_contact1',
       'sen_contact2', 'sen_contact3', 'village_name', 'cluster_mapping']]
Posted_field_staff={
    'Nema Kuta':'Mamadou Jarjusey',
    'Nema':' Lamin Njai',
    'Kwinella Sansankono':'Modou Lamin Njai',
    'Kwinella Nia-Kunda':'Foday K Darbo',
    'Dumbuto':'Ousman Kambi',
    'Niorro Jattaba': 'Modou Bah',
    'Jiffarong':'Lamin Jatta',
    'Kuli Kunda':' Tumbulu Drammeh',
    'keneba':'Lamin LK Jawla',
    'Manduar':' Abdoulie Jallow',
    'Karantaba':'Demba Jallow',
    'Kemoto': 'Amadou Jatta',
    'MASSEMBEH': 'Sidu Sibi',
    'KAIAF': 'Foday Sanyang',
    'Kolior':' Alieu Bah'
}

sen_log_data['staff']=sen_log_data['cluster_mapping'].map(Posted_field_staff)



In [7]:
#all eligible
eligible=df.export_report(report_id='9786')
df_main_eligible=pd.DataFrame(eligible)

In [8]:
lykebba=pd.read_csv('ly_kebba.csv')

In [9]:

consent_v4_Ab=df.export_records(forms=['consent'])
consent_v4_Ab=pd.DataFrame(consent_v4_Ab)
consent_v4_Ab=consent_v4_Ab[
    (consent_v4_Ab['cconsent']=='1')

]

consent_v4_Ab=consent_v4_Ab[['participant_id', 'cconsent', 'consent_form_date_q10']]
consent_v4_Ab= consent_v4_Ab.sort_values(by=['participant_id','consent_form_date_q10'])
consent_v4_Ab= consent_v4_Ab.drop_duplicates(subset='participant_id', keep='last')
consent_v4_Ab=consent_v4_Ab[
   (consent_v4_Ab['consent_form_date_q10']>='2023-12-19')
]



In [10]:
df_main_eligible_filterd=pd.merge(df_main_eligible,consent_v4_Ab,on='participant_id',how='inner').drop_duplicates(subset='participant_id')

In [11]:
df_main_eligible_filterd=df_main_eligible_filterd[['participant_id','cconsent']]


In [12]:
ck_v4_Ab=df.export_report(report_id='10375')
ck_v4_Ab=pd.DataFrame(ck_v4_Ab)
ck_v4_Ab_merg=pd.merge(ck_v4_Ab,df_main_eligible_filterd,on='participant_id',how='inner').drop_duplicates(subset='participant_id')
ck_v4_Ab_merg=ck_v4_Ab_merg[['participant_id','ck_wkno']]

In [13]:

HCG_Positive_AB=df.export_report(report_id='9990')
HCG_Positive_AB=pd.DataFrame(HCG_Positive_AB)


#Positive Booking Scan
Booking_Scan_AB=df.export_report(report_id='9991')
Booking_Scan_AB=pd.DataFrame(Booking_Scan_AB)

#withdrawal
withdrawal_AB=df.export_report(report_id='9989')
withdrawal_AB=pd.DataFrame(withdrawal_AB)

additional_dfs = [HCG_Positive_AB,Booking_Scan_AB,withdrawal_AB]
df_combined = pd.concat(additional_dfs)
df_combined_no_duplicates = df_combined.drop_duplicates()
   

In [14]:
   # Remove rows from df_main that have common IDs
ck_v4_Ab_merg= ck_v4_Ab_merg[~ck_v4_Ab_merg['participant_id'].isin(df_combined_no_duplicates['participant_id'])]


In [15]:
maternal_blood_sample_collection_date=df.export_records(forms=['preganancy_urine_collection'])
maternal_blood_sample_collection_date=pd.DataFrame(maternal_blood_sample_collection_date)
maternal_blood_sample_collection_date=maternal_blood_sample_collection_date[['participant_id','uricycle','urine_date']]
maternal_blood_sample_collection_date=maternal_blood_sample_collection_date.rename(columns={'participant_id': 'participant_id_x'})

import pandas as pd
# Remove duplicates based on 'participant_id' and keep the maximum 'matbld_visitdate_q1_e327e1'
maternal_blood_sample_collection_date.drop_duplicates(subset='participant_id_x', keep='last', inplace=True)

# Convert the 'matbld_visitdate_q1_e327e1' column to datetime type (if it's not already)
maternal_blood_sample_collection_date['urine_date'] = pd.to_datetime(maternal_blood_sample_collection_date['urine_date'])

# Sort the DataFrame based on 'matbld_visitdate_q1_e327e1' in descending order
maternal_blood_sample_collection_date.sort_values(by='urine_date', ascending=False, inplace=True)
maternal_blood_sample_collection_date=maternal_blood_sample_collection_date.rename(columns={'participant_id_x': 'participant_id'})
# Reset the index after sorting
maternal_blood_sample_collection_date.reset_index(drop=True, inplace=True)


In [16]:
df_callist=pd.merge( ck_v4_Ab_merg,sen_log_data,on='ck_wkno',how='left')

In [17]:
data_lmp=df.export_records(forms=['lmp'])
data_lmp=pd.DataFrame(data_lmp)
data_lmp=data_lmp[
    (data_lmp['lmp_complete']=='2')
]
data_lmp=data_lmp[['participant_id','lmp_cycle','lmp_form_date_q5','lmp_temp_exclude___1','lmp_temp_exclude___2','lmp_temp_exclude___3','lmp_temp_exclude___4','lmp_temp_exclude_oth','lmp_complete']]
data_lmp_last_visit=data_lmp[['participant_id','lmp_cycle','lmp_form_date_q5']]
data_lmp_last_visit = data_lmp_last_visit.sort_values(by=['participant_id', 'lmp_form_date_q5'])
data_lmp_last_visit=data_lmp_last_visit.rename(columns={'lmp_cycle':'last LMP cycle Visit'})
data_lmp_last_visit['lmp_form_date_q5'] = pd.to_datetime(data_lmp_last_visit['lmp_form_date_q5'])
# Calculate the next cycle due date by adding 28 days
data_lmp_last_visit['next_cycle_due_date'] = data_lmp_last_visit['lmp_form_date_q5'] + pd.Timedelta(days=28)

data_lmp_last_visit= data_lmp_last_visit.drop_duplicates(subset='participant_id', keep='last')


In [18]:
data_lmp_temp_exclusion=data_lmp[['participant_id','lmp_form_date_q5','lmp_temp_exclude___1','lmp_temp_exclude___2','lmp_temp_exclude___3','lmp_temp_exclude___4','lmp_temp_exclude_oth']]
data_lmp_temp_exclusion=data_lmp_temp_exclusion[
    (data_lmp_temp_exclusion['lmp_temp_exclude___1']=='1')|
    (data_lmp_temp_exclusion['lmp_temp_exclude___2']=='1')|
    (data_lmp_temp_exclusion['lmp_temp_exclude___3']=='1')|
    (data_lmp_temp_exclusion['lmp_temp_exclude___4']=='1')
]
lmp_temp_exclude___1 = {
'1':'Single and not intending to get pregnant'}
data_lmp_temp_exclusion['lmp_temp_exclude___1'] = data_lmp_temp_exclusion['lmp_temp_exclude___1'].map(lmp_temp_exclude___1)

lmp_temp_exclude___2 = {
'1':'Husband away and not expected back soon'}
data_lmp_temp_exclusion['lmp_temp_exclude___2'] = data_lmp_temp_exclusion['lmp_temp_exclude___2'].map(lmp_temp_exclude___2)

lmp_temp_exclude___3 = {
'1':'With a child under 12 months'}
data_lmp_temp_exclusion['lmp_temp_exclude___3'] = data_lmp_temp_exclusion['lmp_temp_exclude___3'].map(lmp_temp_exclude___3)

lmp_temp_exclude___4 = {
'1':'Other'}
data_lmp_temp_exclusion['lmp_temp_exclude___4'] = data_lmp_temp_exclusion['lmp_temp_exclude___4'].map(lmp_temp_exclude___4)
data_lmp_temp_exclusion= data_lmp_temp_exclusion.drop_duplicates('participant_id')

In [19]:
data_lmp_temp_exclusion=data_lmp_temp_exclusion[['participant_id']]
data_lmp_temp_exclusion['Lmp_Status']='Temporal Excluded'

In [20]:
indigo_consent=indigo.export_records(forms=['consent','ultrasound_scan'])
indigo_consent=pd.DataFrame(indigo_consent)
indigo_consent=indigo_consent[['con_participant_eden_num_q11','pregnancy_confirmed_by_ult_q2']]
indigo_consent=indigo_consent.rename(columns={'con_participant_eden_num_q11':'participant_id'})
indigo_consent=indigo_consent[
    (indigo_consent['participant_id']!='')&
    (indigo_consent['pregnancy_confirmed_by_ult_q2']=='1')
]

In [21]:
df_eden_ultrasound_scan=df.export_records(forms=['ultrasound_scan'])
df_eden_ultrasound_scan=pd.DataFrame(df_eden_ultrasound_scan)
df_offpeak=df_eden_ultrasound_scan[
    (df_eden_ultrasound_scan['offpeak_season']=='1')
]
df_offpeak[['participant_id','offpeak_season']]


,participant_id,offpeak_season
1431,EDN0545-E,1


In [22]:
df_callist_cycle_20=pd.merge(df_callist,maternal_blood_sample_collection_date,on='participant_id',how='left')
df_callist_cycle_27=pd.merge(df_callist_cycle_20,data_lmp_temp_exclusion,on='participant_id',how='left')
df_callist_cycle_28=pd.merge(df_callist_cycle_27,data_lmp_last_visit,on='participant_id',how='left')

df_callist_cycle_28=df_callist_cycle_28[~df_callist_cycle_28['participant_id'].isin(indigo_consent['participant_id'])]
#df_callist_cycle_27=pd.concat(df_callist_cycle_27)
#df_callist_cycle_20= df_callist_cycle_20[~df_callist_cycle_20['participant_id'].isin(data_lmp_temp_exclusion['participant_id'])]
df_callist_cycle_28.to_csv('cycle_round_callist.csv',index=False)

In [23]:
df_lmp2=df.export_records(forms=['lmp'])
df_lmp2=pd.DataFrame(df_lmp2)
df_lmp2=df_lmp2[
    (df_lmp2['lmp_form_date_q5']!='')
]

df_lmp2_merg=pd.merge(df_lmp2,ck_v4_Ab_merg,on='participant_id',how='left')

df_lmp2_merg1=pd.merge(df_lmp2_merg,sen_log_data,on='ck_wkno',how='left')
df_lmp2_merg1.to_csv('LMP_progress.csv',index=False)


In [24]:
df_booking=pd.read_csv('booking_placenta.csv')
booking_marg=pd.merge(df_booking,ck_v4_Ab, on='participant_id', how='inner').drop_duplicates('participant_id')
booking_marg1=pd.merge(booking_marg,sen_log_data,on='ck_wkno',how='inner')
booking_marg1.to_csv('booking_marg1.csv',index=False)

In [25]:
#df_callist_1=df_callist_cycle_27[~df_callist_cycle_27['participant_id'].isin(weekly_round_call['participant_id'])]
#df_callist_1.to_csv('df_cycle_30.csv',index=False)

In [26]:
#import win32com.client as win32

#def send_outlook_email(subject, body, to_recipients, cc_recipients=None, attachment=None):
  #  try:
        # Attempt to get an instance of the Outlook application
 #       outlook = win32.GetActiveObject("Outlook.Application")
  #  except:
        # If Outlook is not running, start a new instance
  #      outlook = win32.Dispatch("Outlook.Application")

  #  mail = outlook.CreateItem(0)  # 0 means the email is a new mail item

  #  mail.Subject = subject
   # mail.Body = body

    # Adding recipients
  #  mail.To = ";".join(to_recipients)  # ";" is used to separate multiple recipients

   # if cc_recipients:
    #    mail.CC = ";".join(cc_recipients)

    # Attach a file if specified
   # if attachment:
     #   mail.Attachments.Add(attachment)

    # Display the email before sending (Outlook will open if not already running)
  #  mail.Display()

    # Sending the email
   # mail.Send()
   # print("Email sent successfully!")

#if __name__ == "__main__":
    # Example usage:
    #subject = "cycle Round callist"
   # body = "This is a test email sent using Python and Outlook."
   # to_recipients = ["lshab52@lshtm.ac.uk", "abliebah@mrc.gm"]
   # cc_recipients = ["lshab52@lshtm.ac.uk", "abliebah@mrc.gm"]
   # attachment = r"C:\Users\abliebah\OneDrive - London School of Hygiene and Tropical Medicine\Desktop\Ex_Files_Python_Essential_Training\Exercise Files\Idea3\Idea\eden\cycle_round_callist.csv" 

   # send_outlook_email(subject, body, to_recipients, cc_recipients, attachment)


In [27]:
nioro_jattaba=df.export_records(forms=['consent'])
nioro_jattaba=pd.DataFrame(nioro_jattaba)
nioro_jattaba=nioro_jattaba[['participant_id','ck_wkno','cconsent','redcap_repeat_instance','consent_form_date_q10']]
nioro_jattaba=nioro_jattaba[
  (nioro_jattaba['consent_form_date_q10']!='')
]

nioro_jattaba= nioro_jattaba.sort_values(by=['participant_id','consent_form_date_q10'])
nioro_jattaba=nioro_jattaba.drop_duplicates(subset='participant_id', keep='last')

nioro_jattaba=nioro_jattaba[
    (nioro_jattaba['consent_form_date_q10']<'2023-12-19')
]

nioro_jattaba=nioro_jattaba.rename(columns={'ck_wkno':'ck_wkno'})
nioro_jattaba_marg=pd.merge(sen_log_data,nioro_jattaba,on='ck_wkno', how='inner')
nioro_jattaba_marg.to_csv('nioro_jattaba_marg_eligible.csv',index=False)

In [34]:
read_excel_1=pd.read_excel('EDEN_dry_season_2025_1yr_bleeds.xlsx')
read_excel_1=pd.DataFrame(read_excel_1)
read_excel_2=pd.merge(read_excel_1,ck_v4_Ab,on='participant_id',how='left')
read_excel_3=pd.merge(read_excel_2,sen_log_data,on='ck_wkno',how='left').drop_duplicates('participant_id')
read_excel_3.to_csv('marge_EDEN_dry_season_2025_1yr_bleeds.csv',index=False) 
read_excel_1


,participant_id,placenta_available,del_ddate_q7,sbc_dry_coll_date,sbc_rainy_coll_date,con_participantid_q1_y,ind_6mth_bleed,one.year.blood.status,collect.dry.2025
0,EDN0078-D,1,2024-01-08,NaN,Use INDiGO 6mth bleed,IN-M-081D,2024-07-08,rainy only,True
1,EDN0188-F,1,2025-02-25,NaN,Use INDiGO 6mth bleed,IN-M-475G,2025-08-25,rainy only,True
2,EDN0199-E,1,2024-01-19,NaN,Use INDiGO 6mth bleed,IN-M-071F,2024-07-19,rainy only,True
3,EDN0206-C,1,2024-01-25,NaN,Use INDiGO 6mth bleed,IN-M-067H,2024-07-25,rainy only,True
4,EDN0224-E,1,2023-11-20,NaN,2024-09-25 00:00:00,NaN,NaT,rainy only,True
...,...,...,...,...,...,...,...,...,...
57,EDN2588-G,1,2025-01-07,NaN,Use INDiGO 6mth bleed,IN-M-379Q,2025-07-07,rainy only,True
58,EDN2661-X,1,2025-02-05,NaN,Use INDiGO 6mth bleed,IN-M-425E,2025-08-05,rainy only,True
59,EDN2696-T,1,2025-01-13,NaN,Use INDiGO 6mth bleed,IN-M-456A,2025-07-13,rainy only,True
60,EDN2707-W,1,2025-01-21,NaN,Use INDiGO 6mth bleed,IN-M-380A,2025-07-21,rainy only,True
